# VibeScent Week 2 Pipeline -- Harsh Agarwal
**P0 stages** (must-ship, ~60 min on A100) + **P1 stages** (spillover OK, ~2.5 additional hours)

Resumable via Drive checkpointing. GPU-tier auto-detection (A100/L4/T4).

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  Stage 1 · Setup  ·  VibeScent Week 2 Pipeline  ·  Harsh Agarwal       ║
# ║  Hyper-optimised: uv (50-100× faster than pip) · depth-1 git clone     ║
# ║  Parallel Drive mount · HF cache on Drive · tqdm ETAs on every step    ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import os, sys, time, subprocess, importlib.util, threading, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

# ─── 0. Bootstrap tqdm FIRST (needed for all progress bars below) ─────────
try:
    from tqdm.auto import tqdm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "tqdm", "-q"],
                   check=True, capture_output=True)
    from tqdm.auto import tqdm

# ─── Constants ────────────────────────────────────────────────────────────
REPO_URL         = "https://github.com/GavinGongTech/VibeScent.git"
BRANCH           = "main"
REPO_DIR         = Path("/content/vibescent")
PIPELINE_VERSION = "v1"

IS_COLAB  = os.path.exists("/content") or "google.colab" in sys.modules
IS_KAGGLE = os.path.exists("/kaggle/working") or os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None
if IS_KAGGLE:
    REPO_DIR = Path("/kaggle/working/vibescent")

_CYAN    = "\033[96m"
_GREEN   = "\033[92m"
_YELLOW  = "\033[93m"
_RED     = "\033[91m"
_DIM     = "\033[2m"
_RESET   = "\033[0m"

# ─── Master progress bar (8 named steps, ETA shown at all times) ───────────
_STEPS = [
    "Clone / update repo",
    "Bootstrap uv",
    "Install vibescents",
    "Install Colab GPU deps",
    "Mount Drive + secrets",
    "HF cache → Drive",
    "Detect GPU + disk",
    "Validate imports",
]
BAR_FMT = (
    "{desc:<42} {percentage:3.0f}%|{bar:22}| "
    "{n_fmt}/{total_fmt} [{elapsed}<{remaining}]"
)
_master = tqdm(
    total=len(_STEPS),
    bar_format=BAR_FMT,
    colour="cyan",
    dynamic_ncols=True,
)
_t_start = time.perf_counter()

def _tick(label: str) -> None:
    _master.set_description(f"{_CYAN}⚙{_RESET}  {label}")
    _master.refresh()

def _done(label: str, detail: str = "") -> None:
    elapsed = f"({time.perf_counter() - _t_start:.1f}s)"
    line = f"  {_GREEN}✓{_RESET} {label}"
    if detail: line += f"  {_DIM}{detail}{_RESET}"
    line += f"  {_DIM}{elapsed}{_RESET}"
    tqdm.write(line)
    _master.update(1)

def _warn(msg: str) -> None:
    tqdm.write(f"  {_YELLOW}⚠{_RESET}  {msg}")

def _fail(msg: str) -> None:
    tqdm.write(f"  {_RED}✗{_RESET}  {msg}")

def _run(cmd: list, *, cwd=None, timeout=180) -> str:
    """Run a subprocess; raise RuntimeError with last 2 KB of stderr on failure."""
    r = subprocess.run(cmd, capture_output=True, text=True,
                       cwd=str(cwd) if cwd else None, timeout=timeout)
    if r.returncode != 0:
        raise RuntimeError(f"$ {' '.join(cmd)}\n{r.stderr[-2000:]}")
    return r.stdout.strip()

def _install_with_bar(cmd: list, desc: str = "packages") -> None:
    """
    Stream subprocess stdout line-by-line and count installs in a sub-bar.
    Works with both pip ("Collecting foo") and uv ("Installed X packages").
    """
    sub = tqdm(
        desc=f"    {desc}",
        unit="pkg",
        bar_format="    {desc}: {n:>3} pkg [{elapsed}, {rate_fmt}]",
        leave=False,
        colour="green",
    )
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
    log_lines: list[str] = []
    for line in iter(proc.stdout.readline, ""):
        log_lines.append(line)
        low = line.lower().strip()
        # pip emits "Collecting X" / "Installing collected packages: ..."
        # uv  emits "Installed X packages in Y" / "   + foo 1.2.3"
        if low.startswith(("collecting ", "   + ", "installing collected", "installed ")):
            sub.update(1)
    proc.wait()
    sub.close()
    if proc.returncode != 0:
        raise RuntimeError("Install failed:\n" + "".join(log_lines[-40:]))

# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 · Clone / update repo
# Technique: --depth 1 --single-branch skips full git history → ~10× faster.
# On re-run: fetch + reset instead of re-cloning (instant if up-to-date).
# ══════════════════════════════════════════════════════════════════════════════
_tick("Clone / update repo")

if IS_COLAB or IS_KAGGLE:
    if not REPO_DIR.exists():
        tqdm.write(f"    → Cloning (depth=1, single-branch {BRANCH})...")
        _run(["git", "clone",
              "--depth",         "1",       # only HEAD, not full history
              "--single-branch",            # only the one branch
              "--branch",        BRANCH,
              REPO_URL,          str(REPO_DIR)],
             timeout=120)
    else:
        # Already cloned — fast-forward to latest without re-downloading
        _run(["git", "fetch", "--depth", "1", "origin", BRANCH],
             cwd=REPO_DIR, timeout=60)
        _run(["git", "reset", "--hard", f"origin/{BRANCH}"],
             cwd=REPO_DIR, timeout=30)
    os.chdir(str(REPO_DIR))
    commit = _run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR)
    _done("Repo ready", f"HEAD={commit}")
else:
    # Local dev — already in the right directory
    os.chdir(str(Path(__file__).parent.parent) if "__file__" in dir() else str(Path.cwd()))
    _done("Local dev — skipping clone")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 · Bootstrap uv
# Technique: uv (Rust) resolves + installs packages 50-100× faster than pip.
# One-time cost: pip install uv ≈ 3 s.  All subsequent installs use uv.
# Fallback: if uv bootstrap fails, we fall back to plain pip.
# ══════════════════════════════════════════════════════════════════════════════
_tick("Bootstrap uv")

_UVC = None   # will hold "uv" or None (pip fallback)
try:
    ver = _run(["uv", "--version"])
    _UVC = "uv"
    _done("uv already available", ver)
except (FileNotFoundError, RuntimeError):
    try:
        tqdm.write("    → pip install uv (one-time ~3 s)…")
        subprocess.run([sys.executable, "-m", "pip", "install", "uv", "-q"],
                       check=True, capture_output=True, timeout=60)
        ver = _run(["uv", "--version"])
        _UVC = "uv"
        _done("uv installed", ver)
    except Exception as e:
        _warn(f"uv unavailable ({e}); falling back to pip")
        _master.update(1)

def _pip(args: list, desc: str = "packages") -> None:
    """Install via uv (preferred) or pip (fallback)."""
    if _UVC:
        # --system: install into the active Python environment (no venv)
        # -q:       suppress uv noise; sub-bar handles progress reporting
        cmd = [_UVC, "pip", "install", "--system"] + args
    else:
        cmd = [sys.executable, "-m", "pip", "install"] + args
    _install_with_bar(cmd, desc=desc)

# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 · Install vibescents package
# Technique: importlib.util.find_spec to skip on re-run (saves ~5 s).
# Editable install (-e .) so in-notebook changes to src/ are live immediately.
# ══════════════════════════════════════════════════════════════════════════════
_tick("Install vibescents")

_already = importlib.util.find_spec("vibescents") is not None
if not _already or IS_COLAB:
    tqdm.write("    → pip install -e . (editable, changes to src/ are live)…")
    if _UVC:
        _run([_UVC, "pip", "install", "--system", "-e", "."], timeout=120)
    else:
        _run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], timeout=120)
    _done("vibescents installed (editable)")
else:
    _done("vibescents already in env — skipped", "importlib.util.find_spec hit")

# Force-add src/ to sys.path — uv --system install writes to a different
# site-packages than Colab's kernel picks up. This is the reliable fallback.
_src_path = str(REPO_DIR / "src")
if _src_path not in sys.path:
    sys.path.insert(0, _src_path)
    tqdm.write(f"    → sys.path patched: {_src_path}")

# ══════════════════════════════════════════════════════════════════════════════
# -- Early GPU probe -- detect_gpu_tier() uses torch VRAM which is always
# available on Kaggle/Colab; shares the same classification as USE_LIGHTWEIGHT_LLM
# so both variables always agree on which requirements file gets installed.
from vibescents.week2_pipeline import detect_gpu_tier as _detect_gpu_tier_early
try:
    _IS_T4_EARLY = _detect_gpu_tier_early() == "T4"
except RuntimeError:
    _IS_T4_EARLY = False

# STEP 4 · Install Colab GPU deps  (vllm, outlines, sentence-transformers…)
# Technique: uv resolves all deps in one shot with parallel downloads.
# requirements.colab.txt is version-pinned so installs are reproducible.
# Skipped on local dev — no GPU deps needed for linting/testing.
# ══════════════════════════════════════════════════════════════════════════════
_tick("Install Colab GPU deps")

_req    = REPO_DIR / "notebooks/requirements.colab.txt"
_req_t4 = REPO_DIR / "notebooks/requirements.colab.t4.txt"

if (IS_COLAB or IS_KAGGLE) and _req.exists():
    if _IS_T4_EARLY and _req_t4.exists():
        tqdm.write(f"    → T4 detected: {_req_t4.name} (transformers>=5.5, no vLLM)…")
        _pip(["-r", str(_req_t4)], desc="T4 GPU deps")
    else:
        tqdm.write(f"    → Installing from {_req.name} (parallel resolution)…")
        _pip(["-r", str(_req)], desc="GPU deps")
    _done("GPU deps installed")
else:
    _done("Colab deps skipped (local dev or file missing)")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 5 · Mount Google Drive + load Colab secrets  (run in parallel)
# Technique: Drive mount and secret loading are both I/O-bound; running them
# concurrently with threading.Thread shaves ~3-5 s off total setup time.
# ══════════════════════════════════════════════════════════════════════════════
_tick("Mount Drive + secrets")

DRIVE_BASE = "artifacts"   # default for local dev
GEMINI_KEY = VOYAGE_KEY = None
_drive_exc = None

def _mount():
    global DRIVE_BASE, _drive_exc
    try:
        from google.colab import drive as _gd
        _gd.mount("/content/drive", force_remount=False)
        p = Path("/content/drive/MyDrive/vibescent/week2")
        p.mkdir(parents=True, exist_ok=True)
        DRIVE_BASE = str(p)
    except Exception as exc:
        _drive_exc = exc

def _secrets():
    global GEMINI_KEY, VOYAGE_KEY
    try:
        from google.colab import userdata as _ud
        try: GEMINI_KEY = _ud.get("GEMINI_API_KEY")
        except Exception: pass
        try: VOYAGE_KEY = _ud.get("VOYAGE_API_KEY")
        except Exception: pass
    except ImportError:
        pass
    GEMINI_KEY = GEMINI_KEY or os.environ.get("GEMINI_API_KEY")
    VOYAGE_KEY = VOYAGE_KEY or os.environ.get("VOYAGE_API_KEY")
    if GEMINI_KEY: os.environ["GEMINI_API_KEY"] = GEMINI_KEY
    if VOYAGE_KEY: os.environ["VOYAGE_API_KEY"] = VOYAGE_KEY

_t_drive   = threading.Thread(target=_mount,   daemon=False)
_t_secrets = threading.Thread(target=_secrets, daemon=False)
_t_drive.start();   _t_secrets.start()
_t_drive.join(timeout=50)
if _t_drive.is_alive():
    _drive_exc = TimeoutError("Drive mount timed out after 50s — run from Colab web UI, not VS Code/local")
_t_secrets.join(timeout=5)

if _drive_exc and IS_COLAB:
    _warn(f"Drive mount failed — using local artifacts/  [{_drive_exc}]")
    print("=" * 60)
    print("WARNING: Drive mount failed! Artifacts will be saved to")
    print(f"  {DRIVE_BASE}")
    print("These WILL BE LOST if the Colab session disconnects!")
    print("=" * 60)

_key_status = (
    f"gemini={'✓' if GEMINI_KEY else '✗'}  "
    f"voyage={'✓' if VOYAGE_KEY else '✗'}"
)
_done("Drive + secrets", f"base={DRIVE_BASE}  ·  {_key_status}")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 6 · Redirect HuggingFace cache → Drive
# Technique: On Colab, the default HF cache (/root/.cache) is wiped when the
# runtime disconnects.  Pointing HF_HOME at Drive means downloaded model
# weights (Qwen3-Embedding-8B ≈ 16 GB) survive disconnects → saves 10-15 min
# on reconnect.  Only set after Drive is mounted (previous step).
# ══════════════════════════════════════════════════════════════════════════════
_tick("HF cache → Drive")

if IS_COLAB and DRIVE_BASE.startswith("/content/drive"):
    HF_CACHE = str(Path(DRIVE_BASE).parent.parent / "hf_cache")
    Path(HF_CACHE).mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"]           = HF_CACHE
    os.environ["HF_DATASETS_CACHE"]  = str(Path(HF_CACHE) / "datasets")
    _done("HF cache → Drive", HF_CACHE)
else:
    HF_CACHE = os.path.expanduser("~/.cache/huggingface")
    _done("HF cache stays local (no Drive)")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 7 · Detect GPU tier + disk space
# ══════════════════════════════════════════════════════════════════════════════
_tick("Detect GPU + disk")

import torch
from vibescents.week2_pipeline import detect_gpu_tier, check_disk_space

try:
    GPU_TIER  = detect_gpu_tier()
    _props    = torch.cuda.get_device_properties(0)
    _vram_gb  = _props.total_memory / 1e9
    _gpu_name = _props.name
    _done("GPU detected", f"{GPU_TIER} · {_gpu_name} · {_vram_gb:.1f} GB VRAM")
except RuntimeError:
    GPU_TIER  = "CPU"
    _warn("No CUDA GPU — embedding will be very slow on CPU")

USE_LOCAL_LLM     = GPU_TIER == "A100"
USE_LOCAL_MM      = GPU_TIER in ("A100", "L4", "T4")  # multimodal on all GPU tiers
USE_INT8_VL       = GPU_TIER == "T4"                  # int8 VL embedder on T4
USE_INT8_EMBEDDER = GPU_TIER == "T4"
USE_LIGHTWEIGHT_LLM = GPU_TIER == "T4"   # T4: Gemma 4 E4B int4 replaces Gemini API

_disk      = shutil.disk_usage("/content" if IS_COLAB else ".")
_free_gb   = _disk.free  / 1e9
_total_gb  = _disk.total / 1e9
_used_pct  = 100 * _disk.used / _disk.total
tqdm.write(f"    → Disk: {_free_gb:.1f} GB free / {_total_gb:.1f} GB  ({_used_pct:.0f}% used)")

if IS_COLAB:
    check_disk_space("/content", warn_pct=75, abort_pct=92)

# ══════════════════════════════════════════════════════════════════════════════
# STEP 8 · Validate imports (catch broken installs before Stage 2)
# ══════════════════════════════════════════════════════════════════════════════
_tick("Validate imports")

_modules = [
    "vibescents.week2_pipeline",
    "vibescents.schemas",
    "vibescents.similarity",
    "vibescents.benchmark",
    "vibescents.enrich",
    "vibescents.fusion",
    "vibescents.image_preprocess",
    "vibescents.image_scoring",
]
import importlib
_import_bar = tqdm(
    _modules, desc="    modules",
    bar_format="    {desc}: {n:>2}/{total:>2} [{elapsed}]",
    leave=False, colour="green",
)
for _mod in _import_bar:
    importlib.import_module(_mod)

_done("All modules importable", f"{len(_modules)} checked")

# ─── Finish master bar ────────────────────────────────────────────────────
_master.close()

# ─── Summary table ────────────────────────────────────────────────────────
_elapsed = time.perf_counter() - _t_start
_w = 56
print(f"\n{'━' * _w}")
print(f"  {'Setup complete':40}  {_elapsed:5.1f} s")
print(f"{'━' * _w}")
_rows = [
    ("GPU tier",           GPU_TIER),
    ("Local LLM (A100)",   str(USE_LOCAL_LLM)),
    ("Local MM (A100/L4)", str(USE_LOCAL_MM)),
    ("Int8 embedder (T4)", str(USE_INT8_EMBEDDER)),
    ("Lightweight LLM (T4)", str(USE_LIGHTWEIGHT_LLM)),
    ("Drive base",         DRIVE_BASE),
    ("HF cache",           HF_CACHE),
    ("Gemini key",         "set ✓" if GEMINI_KEY else "NOT SET ✗"),
    ("Voyage key",         "set ✓" if VOYAGE_KEY else "not set"),
    ("Pipeline version",   PIPELINE_VERSION),
    ("Package manager",    "uv" if _UVC else "pip (uv unavailable)"),
]
for k, v in _rows:
    print(f"  {k:<28} {v}")
print(f"{'━' * _w}")

In [ ]:
# === Stage 2: Sanity Check ===
import time
from tqdm.auto import tqdm
import pandas as pd
from pathlib import Path

_t0 = time.perf_counter()
steps = ["Load CSV", "Row count assert", "File checks", "Spot-check", "Column check"]
bar = tqdm(steps, desc="Stage 2  --  Sanity", ncols=90, unit="check")

bar.set_postfix_str("Loading CSV...")
bar.update(0)
df = pd.read_csv("data/processed/vibescent_unified.csv")
bar.update(1)

if len(df) != 35889:
    print(f"WARNING: expected 35889 rows, got {len(df)} -- continuing anyway")
bar.update(1)

assert Path("examples/occasions.json").exists(), "occasions.json missing"
assert Path("examples/benchmark_briefs.json").exists(), "benchmark_briefs.json missing"
bar.update(1)

sample = df.sample(5, random_state=42)[["name", "brand", "top_notes"]].to_string()
bar.update(1)

text_col = "embedding_text" if "embedding_text" in df.columns else "retrieval_text"
if text_col not in df.columns:
    print("WARNING: no embedding_text/retrieval_text column  --  will construct on-the-fly")
bar.update(1)
bar.close()

print(f"Corpus: {len(df):,} rows | text_col={text_col!r} | {time.perf_counter()-_t0:.1f}s")
print(f"Columns: {list(df.columns)}")
print(f"\nSample rows:\n{sample}")


In [ ]:
# === Stage 3: Load Qwen3-Embedding-8B ===
PIPELINE_VERSION = globals().get("PIPELINE_VERSION", "v1")  # guard for re-run safety
import time
import torch
from tqdm.auto import tqdm
from vibescents.week2_pipeline import ensure_model_loaded, stage_complete, check_disk_space
import numpy as np
from pathlib import Path

ARTIFACTS_RAW = Path(DRIVE_BASE) / "fragrance_raw_full"

if stage_complete("embed_raw", ARTIFACTS_RAW, PIPELINE_VERSION):
    print("Stage 3: embeddings already complete  --  skipping model load")
    embedder = None
else:
    steps = ["Import SentenceTransformer", "Build model_kwargs", "Load model", "Pre-flight encode", "Dim assert"]
    bar = tqdm(steps, desc="Stage 3  --  Load embedder", ncols=90, unit="step")
    _t0 = time.perf_counter()

    bar.set_postfix_str("importing...")
    from sentence_transformers import SentenceTransformer
    bar.update(1)

    model_kwargs = {"torch_dtype": "float16"}
    if USE_INT8_EMBEDDER:
        model_kwargs = {"load_in_8bit": True}  # torch_dtype conflicts with bitsandbytes int8
    else:
        # SDPA: PyTorch-native, no flash_attn install needed, works on all GPUs
        model_kwargs["attn_implementation"] = "sdpa"
    bar.update(1)

    bar.set_postfix_str("downloading/loading weights (~16 GB fp16)...")
    embedder = SentenceTransformer(
        "Qwen/Qwen3-Embedding-8B",
        trust_remote_code=True,
        model_kwargs=model_kwargs,
    )
    ensure_model_loaded("qwen3-embed-8b", lambda: None)
    bar.update(1)

    bar.set_postfix_str("pre-flight encode...")
    test_emb = embedder.encode(
        ["test sentence one", "test sentence two"],
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    bar.update(1)

    assert test_emb.shape[1] == 4096, f"Expected 4096 dims, got {test_emb.shape[1]}"
    bar.update(1)
    bar.close()

    print(f"Qwen3-Embedding-8B ready | dim={test_emb.shape[1]} | {time.perf_counter()-_t0:.1f}s")
    if os.path.exists("/content"):
        check_disk_space("/content")



In [ ]:
# === Stage 4: Embed Full 35,889 Corpus at 1024 dim ===
PIPELINE_VERSION = globals().get("PIPELINE_VERSION", "v1")  # guard for re-run safety
# Speed-ups:  batch_size=64 (+2x), torch.autocast (+1.3x), background Drive save
import time
import threading
import subprocess
import torch
import numpy as np
from tqdm.auto import tqdm
from pathlib import Path
from vibescents.week2_pipeline import (
    embed_corpus_resume, embedding_sanity_check, write_manifest, stage_complete,
)

# Re-check Drive at stage start: setup may have timed out but user manually mounted
if DRIVE_BASE == "artifacts" and Path("/content/drive/MyDrive/vibescent/week2").exists():
    DRIVE_BASE = "/content/drive/MyDrive/vibescent/week2"
    tqdm.write("    → Drive now accessible — switching artifacts to Drive")
ARTIFACTS_RAW = Path(DRIVE_BASE) / "fragrance_raw_full"
_t0 = time.perf_counter()

if stage_complete("embed_raw", ARTIFACTS_RAW, PIPELINE_VERSION):
    print("Stage 4: loading cached embeddings")
    raw_embeddings = np.load(ARTIFACTS_RAW / "embeddings.npy")
    print(f"Loaded: {raw_embeddings.shape} in {time.perf_counter()-_t0:.1f}s")
else:
    if embedder is None:
        raise RuntimeError(
            "Embedder is None -- re-run Stage 3 (Cell 4) to load Qwen3-Embedding-8B first."
        )

    # -- Pick text column ------------------------------------------------------
    if "embedding_text" in df.columns:
        texts = df["embedding_text"].fillna("").tolist()
    elif "retrieval_text" in df.columns:
        texts = df["retrieval_text"].fillna("").tolist()
    else:
        def _col(c):
            return df[c].fillna("") if c in df.columns else pd.Series([""] * len(df))
        texts = (
            _col("name") + " " + _col("brand") + " " +
            _col("main_accords") + " " + _col("top_notes") + " " +
            _col("middle_notes") + " " + _col("base_notes")
        ).tolist()

    # -- Resume check ---------------------------------------------------------
    CKPT_DIR = Path(DRIVE_BASE) / "embed_raw_ckpt"
    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    partial, resume_batch = embed_corpus_resume(CKPT_DIR)
    if partial is not None:
        print(f"Resuming from batch {resume_batch} ({partial.shape[0]} rows done)")

    # -- Optimized encode loop -------------------------------------------------
    BATCH = 64          # 2x original  --  A100 has headroom at fp16
    CKPT_EVERY = 50     # checkpoint every 50 batches ~= 3,200 rows
    n_batches = (len(texts) + BATCH - 1) // BATCH
    parts = []
    _save_thread = None

    pbar = tqdm(
        range(resume_batch, n_batches),
        desc="Stage 4  --  Embed corpus",
        unit="batch",
        total=n_batches - resume_batch,
        ncols=100,
        dynamic_ncols=False,
    )

    for batch_idx in pbar:
        start = batch_idx * BATCH
        end   = min(start + BATCH, len(texts))
        batch = texts[start:end]

        with torch.no_grad(), torch.amp.autocast('cuda'):  # no_grad prevents autograd graph build; autocast = mixed precision (+~1.3x)
            raw = embedder.encode(
                batch,
                normalize_embeddings=False,            # we renorm after truncation
                show_progress_bar=False,
                convert_to_numpy=True,
            )
        truncated = raw[:, :1024].astype(np.float32)
        parts.append(truncated)

        done_rows = (batch_idx - resume_batch + 1) * BATCH
        pbar.set_postfix(rows=f"{min(done_rows, len(texts)-resume_batch*BATCH):,}")

        # Background checkpoint save (non-blocking)
        if (batch_idx + 1) % CKPT_EVERY == 0:
            _ckpt_new = np.vstack(parts)
            _ckpt_path = CKPT_DIR / f"embeddings_partial_{batch_idx}.npy"
            def _save_ckpt(new_arr=_ckpt_new, prev_arr=partial, path=_ckpt_path):
                combined = np.vstack([prev_arr, new_arr]) if prev_arr is not None else new_arr
                np.save(path, combined)
            if _save_thread and _save_thread.is_alive():
                _save_thread.join()
            _save_thread = threading.Thread(target=_save_ckpt, daemon=False)
            _save_thread.start()

    if _save_thread:
        _save_thread.join()

    pbar.close()

    # -- Assemble + L2-normalize -----------------------------------------------
    print("Assembling + normalizing...", end=" ", flush=True)
    matrix = np.vstack(parts).astype(np.float32)
    if partial is not None:
        matrix = np.vstack([partial, matrix])
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1.0, norms)
    raw_embeddings = matrix / norms
    print(f"done  --  shape {raw_embeddings.shape}")

    if raw_embeddings.shape[0] != 35889 or raw_embeddings.shape[1] != 1024:
        print(f"WARNING: unexpected embedding shape {raw_embeddings.shape} (expected (35889, 1024)) -- continuing anyway")
    embedding_sanity_check(raw_embeddings)

    # -- Background Drive save -------------------------------------------------
    ARTIFACTS_RAW.mkdir(parents=True, exist_ok=True)
    commit_sha = subprocess.run(
        ["git", "rev-parse", "HEAD"], capture_output=True, text=True
    ).stdout.strip()

    def _bg_save():
        np.save(ARTIFACTS_RAW / "embeddings.npy", raw_embeddings)
        write_manifest(
            ARTIFACTS_RAW / "manifest.json",
            model="Qwen/Qwen3-Embedding-8B",
            commit_sha=commit_sha,
            row_count=raw_embeddings.shape[0],
            dim=raw_embeddings.shape[1],
            pipeline_version=PIPELINE_VERSION,
        )
        print(f"  [bg] Saved {raw_embeddings.shape} -> {ARTIFACTS_RAW}")

    _bg_thread = threading.Thread(target=_bg_save, daemon=False)
    _bg_thread.start()
    # Stage 5 can start immediately while save runs in background
    print(f"Saved (background thread running) | elapsed {time.perf_counter()-_t0:.0f}s")




In [ ]:
# === Stage 5: Embed 8 Occasions ===
PIPELINE_VERSION = globals().get("PIPELINE_VERSION", "v1")  # guard for re-run safety
import json
import os
import time
import threading
import subprocess
import numpy as np
from tqdm.auto import tqdm
from pathlib import Path
from vibescents.week2_pipeline import write_manifest, stage_complete

ARTIFACTS_OCC = Path(DRIVE_BASE) / "occasions"
_t0 = time.perf_counter()

if stage_complete("embed_occ", ARTIFACTS_OCC, PIPELINE_VERSION):
    print("Stage 5: loading cached occasion embeddings")
    occ_embeddings = np.load(ARTIFACTS_OCC / "embeddings.npy")
else:
    if globals().get("embedder") is None:
        raise RuntimeError(
            "Embedder is None -- re-run Stage 3 (Cell 4) to load Qwen3-Embedding-8B first."
        )

    # Wait for Stage 4 bg save to finish (if still running)
    try:
        _bg_thread.join()
    except NameError:
        pass

    with open("examples/occasions.json") as f:
        occasions = json.load(f)
    occ_texts = [occ["text"] if isinstance(occ, dict) else str(occ) for occ in occasions]

    pbar = tqdm(occ_texts, desc="Stage 5  --  Occasion embeds", ncols=90, unit="occasion")
    parts = []
    with torch.no_grad(), torch.amp.autocast('cuda'):
        for text in pbar:
            pbar.set_postfix_str(text[:40])
            emb = embedder.encode([text], normalize_embeddings=False, show_progress_bar=False, convert_to_numpy=True)
            trunc = emb[:, :1024].astype(np.float32)
            parts.append(trunc)
    pbar.close()

    occ_embeddings = np.vstack(parts)
    norms = np.linalg.norm(occ_embeddings, axis=1, keepdims=True)
    occ_embeddings = occ_embeddings / np.where(norms == 0, 1.0, norms)

    ARTIFACTS_OCC.mkdir(parents=True, exist_ok=True)
    np.save(ARTIFACTS_OCC / "embeddings.npy", occ_embeddings)
    commit_sha = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    write_manifest(
        ARTIFACTS_OCC / "manifest.json",
        model="Qwen/Qwen3-Embedding-8B",
        commit_sha=commit_sha,
        row_count=occ_embeddings.shape[0],
        dim=occ_embeddings.shape[1],
        pipeline_version=PIPELINE_VERSION,
    )

print(f"Occasions: {occ_embeddings.shape} | {time.perf_counter()-_t0:.1f}s")




In [ ]:
# === Stage 6: Unload Embedder, Load Qwen3.5-27B-GPTQ-Int4 via vLLM ===
import os, time, torch, gc
from tqdm.auto import tqdm
from vibescents.week2_pipeline import ensure_model_loaded, mark_model_unloaded, check_disk_space

_t0 = time.perf_counter()
bar = tqdm(["Unload embedder", "Clear VRAM", "Load LLM", "Smoke test"],
           desc="Stage 6  --  Load LLM", ncols=90, unit="step")

# Step 1: unload embedder
if "embedder" in dir() and embedder is not None:
    del embedder; embedder = None
gc.collect()
bar.update(1)

# Step 2: clear VRAM
torch.cuda.empty_cache()
mark_model_unloaded()
bar.update(1)

if USE_LOCAL_LLM:
    # A100: load Qwen3.5-27B-GPTQ-Int4 via vLLM
    from vibescents.week2_pipeline import smoke_test_enrichment
    import transformers as _tf_check
    _tv = tuple(int(x) for x in _tf_check.__version__.split(".")[:2])
    if _tv < (4, 57):
        raise RuntimeError(
            f"transformers {_tf_check.__version__} is too old (need >=4.57.0 for qwen3_5). "
            "Re-run from cell 1 after a kernel restart — setup upgrades it automatically."
        )

    bar.set_postfix_str("loading Qwen3.5-27B-GPTQ-Int4...")
    from vllm import LLM, SamplingParams

    _llm_engine = LLM(
        model="Qwen/Qwen3.5-27B-GPTQ-Int4",
        quantization="gptq_marlin",
        dtype="float16",
        gpu_memory_utilization=0.88,
        trust_remote_code=True,
    )
    ensure_model_loaded("qwen3.5-27b", lambda: None)

    class VLLMEnrichClient:
        def __init__(self, engine): self._engine = engine
        def generate(self, prompt, schema=None):
            from vibescents.schemas import EnrichmentSchemaV2
            sp_kwargs = {"guided_json": schema} if schema else {}
            sp = SamplingParams(temperature=0.3, max_tokens=512, **sp_kwargs)
            out = self._engine.generate([prompt], sp)
            text = out[0].outputs[0].text.strip()
            try:
                return EnrichmentSchemaV2.model_validate_json(text)
            except Exception:
                from json_repair import repair_json
                return EnrichmentSchemaV2.model_validate_json(repair_json(text))

    llm_client = VLLMEnrichClient(_llm_engine)
    print(f"vLLM engine ready  (Qwen3.5-27B-GPTQ-Int4 · transformers {_tf_check.__version__})")
    bar.update(1)

    # Step 4: smoke test — hard fail, not optional
    bar.set_postfix_str("smoke test...")
    smoke_test_enrichment(llm_client, df.iloc[0])
    print("Smoke test PASSED")
    bar.update(1)

elif USE_LIGHTWEIGHT_LLM:
    # T4: Gemma 4 E4B int4 via transformers + bitsandbytes
    # No vLLM -- vllm pins transformers<5, Gemma 4 needs >=5.5 (mutual conflict)
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    import torch as _torch
    from vibescents.schemas import EnrichmentSchemaV2 as _EnrichSchema
    from json_repair import repair_json as _repair_json

    _GEMMA_MODEL_ID = "google/gemma-4-E4B-it"
    bar.set_postfix_str("loading Gemma 4 E4B int4 (~5.5 GB)...")

    _bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=_torch.float16,  # T4=Turing sm_75: float16, not bfloat16
        bnb_4bit_use_double_quant=True,
    )
    _gemma_tok = AutoTokenizer.from_pretrained(_GEMMA_MODEL_ID)
    _gemma_mdl = AutoModelForCausalLM.from_pretrained(
        _GEMMA_MODEL_ID,
        quantization_config=_bnb_config,
        device_map="auto",
        torch_dtype=_torch.float16,
    )
    _gemma_mdl.eval()
    ensure_model_loaded("gemma-4-e4b", lambda: None)
    print(f"Gemma 4 E4B int4 loaded  ({_GEMMA_MODEL_ID})")
    bar.update(1)

    # SamplingParams shim -- defined at notebook scope so Stages 8/10 can
    # reference it without importing from vllm (conflicts with transformers>=5.5)
    class SamplingParams:
        """Compat shim for vllm.SamplingParams -- avoids vllm import on T4."""
        def __init__(self, *, temperature=0.3, max_tokens=256, guided_json=None):
            self.temperature = temperature
            self.max_tokens = max_tokens
            # guided_json silently ignored: schema constraint is injected via prompt

    class _GemmaOutput:
        def __init__(self, text: str): self.text = text
    class _GemmaReqOutput:
        def __init__(self, text: str): self.outputs = [_GemmaOutput(text)]

    # System prompt: field list derived from the schema so it never drifts;
    # concrete example values instead of '<0.0-1.0>' placeholders which a JSON
    # parser would reject and cause unnecessary json_repair calls.
    _ENRICH_SYSTEM = (
        "You are a luxury fragrance analyst. "
        "Given fragrance details, respond with a JSON object containing exactly "
        f"these keys: {', '.join(_EnrichSchema.model_fields)}.\n"
        'Example: {"likely_season":"fall","likely_occasion":"black tie gala",'
        '"formality":0.9,"fresh_warm":0.2,"day_night":0.85,'
        '"character_tags":["opulent","dark","smoky"],'
        '"vibe_sentence":"A smouldering oriental built for after-dark drama.",'
        '"longevity":"exceptional","projection":"loud",'
        '"mood_tags":["seductive","powerful"],'
        '"color_palette":["ebony","deep amber"]}\n'
        "Output ONLY the JSON. No markdown fences. No explanation."
    )

    _MICRO_BS = 8  # micro-batch: amortises weight reads; ~3-4x vs batch=1 on T4

    class _GemmaEngine:
        """Batch inference engine matching vLLM LLM.generate() call signature."""
        def __init__(self, model, tokenizer, system_prompt: str):
            self._m = model
            self._t = tokenizer
            self._t.padding_side = "left"  # left-pad required for batched causal LM decode
            self._sys = system_prompt

        def generate(self, prompts: list, sp) -> list:
            results = []
            for start in range(0, len(prompts), _MICRO_BS):
                chunk = prompts[start : start + _MICRO_BS]
                # apply_chat_template per-sequence (tokenize=False returns formatted string)
                formatted = [
                    self._t.apply_chat_template(
                        [{"role": "user", "content": self._sys + "\n\nFragrance details:\n" + p}],
                        tokenize=False, add_generation_prompt=True,
                    )
                    for p in chunk
                ]
                # batch-tokenize with left-padding; add_special_tokens=False because
                # apply_chat_template already emitted BOS / special tokens
                enc = self._t(
                    formatted, return_tensors="pt", padding=True,
                    truncation=False, add_special_tokens=False,
                )
                input_ids      = enc["input_ids"].to(self._m.device)
                attention_mask = enc["attention_mask"].to(self._m.device)
                prompt_len     = input_ids.shape[-1]  # same for all (left-padded)
                with _torch.no_grad():
                    out_ids = self._m.generate(
                        input_ids,
                        attention_mask=attention_mask,
                        max_new_tokens=sp.max_tokens,
                        do_sample=sp.temperature > 0,
                        temperature=max(float(sp.temperature), 1e-4),
                        pad_token_id=self._t.eos_token_id,
                    )
                for seq in out_ids:
                    text = self._t.decode(seq[prompt_len:], skip_special_tokens=True).strip()
                    results.append(_GemmaReqOutput(text))
            return results

    _llm_engine = _GemmaEngine(_gemma_mdl, _gemma_tok, _ENRICH_SYSTEM)

    class GemmaEnrichClient:
        def __init__(self, engine): self._engine = engine
        def generate(self, prompt: str, schema=None) -> "_EnrichSchema":
            sp = SamplingParams(temperature=0.3, max_tokens=256)
            out = self._engine.generate([prompt], sp)
            text = out[0].outputs[0].text.strip()
            try:
                return _EnrichSchema.model_validate_json(text)
            except Exception:
                return _EnrichSchema.model_validate_json(_repair_json(text))

    llm_client = GemmaEnrichClient(_llm_engine)

    bar.set_postfix_str("smoke test...")
    from vibescents.week2_pipeline import smoke_test_enrichment
    smoke_test_enrichment(llm_client, df.iloc[0])
    print("Smoke test PASSED")
    bar.update(1)

else:
    # CPU / no-GPU: all LLM stages fall back to Gemini API
    _llm_engine = None
    llm_client = None
    bar.update(1)  # consume 'Load LLM' slot
    bar.update(1)  # consume 'Smoke test' slot
    print("Stage 6: CPU tier -- LLM load skipped, using Gemini API for Stages 7/8/10")

bar.close()

if os.path.exists("/content"):
    check_disk_space("/content")
print(f"Stage 6 complete | {time.perf_counter()-_t0:.0f}s")

In [ ]:
# === Stage 7: Generate 20 Benchmark Labels (3 runs, parallelized) ===
# 3 runs per case are dispatched concurrently via ThreadPoolExecutor
import time
import json
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from pathlib import Path

BENCHMARK_PATH = Path(DRIVE_BASE) / "benchmark_cases.json"
_t0 = time.perf_counter()

if BENCHMARK_PATH.exists():
    with open(BENCHMARK_PATH) as f:
        benchmark_cases = json.load(f)
    print(f"Loaded {len(benchmark_cases)} cached benchmark cases")
else:
    from vibescents.benchmark import BenchmarkGenerator, consolidate_case_drafts

    benchmark_provider = "qwen" if USE_LOCAL_LLM else "gemini"
    with open("examples/benchmark_briefs.json") as f:
        briefs = json.load(f)

    def _gen_runs(brief_obj, n_runs=3):
        # Generate n_runs labels for a single brief (runs in thread).
        from vibescents.benchmark import BenchmarkGenerator, consolidate_case_drafts
        gen = BenchmarkGenerator(provider=benchmark_provider)
        drafts = gen.generate_case_labels(
            case_id=brief_obj["case_id"],
            brief=brief_obj["brief"],
            runs=n_runs,
        )
        return brief_obj["case_id"], consolidate_case_drafts(drafts)

    benchmark_cases = []
    # ThreadPoolExecutor: cases are independent  --  run up to 5 in parallel
    # (Gemini: rate-limit safe at 5; vLLM: engine handles concurrency internally)
    max_workers = 5 if not USE_LOCAL_LLM else 1

    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = {pool.submit(_gen_runs, b): b["case_id"] for b in briefs}
        pbar = tqdm(
            as_completed(futures),
            total=len(futures),
            desc="Stage 7  --  Benchmark labels",
            ncols=90,
            unit="case",
        )
        for fut in pbar:
            case_id, consolidated = fut.result()
            benchmark_cases.append(consolidated.model_dump())
            conf = consolidated.confidence if hasattr(consolidated, "confidence") else "?"
            pbar.set_postfix(last=case_id, conf=f"{conf:.2f}" if isinstance(conf, float) else conf)

    with open(BENCHMARK_PATH, "w") as f:
        json.dump(benchmark_cases, f, indent=2)

valid = sum(1 for c in benchmark_cases if c.get("confidence", 0) >= 0.6)
print(f"Benchmark: {len(benchmark_cases)} cases | {valid} confidence >= 0.6 | {time.perf_counter()-_t0:.0f}s")


In [ ]:
# === Stage 8: Enrich TIER C (500-row sample)  --  vLLM batch inference ===
# Key optimization: vLLM PagedAttention + continuous batching sends ALL prompts
# at once (vs. serial Outlines loop). 5-8x speedup on A100.
import time
import json
import threading
from tqdm.auto import tqdm
import pandas as pd
from pathlib import Path
from vibescents.week2_pipeline import validate_enrichment

TIER_C_PATH = Path(DRIVE_BASE) / "vibescent_enriched_500_v2.csv"
TIER_C_CKPT = str(TIER_C_PATH) + ".ckpt"
_t0 = time.perf_counter()

if TIER_C_PATH.exists():
    enriched_c = pd.read_csv(TIER_C_PATH)
    print(f"Loaded cached TIER C: {len(enriched_c)} rows")
else:
    df_500 = pd.read_csv("data/vibescent_500.csv") if Path("data/vibescent_500.csv").exists() else df.head(500).copy()

    if (USE_LOCAL_LLM or USE_LIGHTWEIGHT_LLM) and globals().get("_llm_engine") and hasattr(globals()["_llm_engine"], "generate"):
        # -- vLLM batch (A100) or Gemma 4 E4B batch (T4) ----------------------
        from vibescents.enrich import _build_prompt
        from vibescents.schemas import EnrichmentSchemaV2
        from json_repair import repair_json
        if USE_LOCAL_LLM:
            from vllm import SamplingParams  # T4 uses shim defined in Cell 6
        import json as _json

        schema_dict = EnrichmentSchemaV2.model_json_schema()
        sp = SamplingParams(
            temperature=0.3,
            max_tokens=256,
            guided_json=schema_dict,  # XGrammar on A100; prompt-injected on T4
        )

        print(f"Building {len(df_500)} prompts...", end=" ", flush=True)
        prompts = [_build_prompt(row) for _, row in df_500.iterrows()]
        print("done")

        _vllm_t = time.perf_counter()
        _batch_label = "vLLM" if USE_LOCAL_LLM else "Gemma"
        print(f"  {_batch_label} batch ({len(prompts)} rows) -- this may take several minutes...")
        try:
            outputs = _llm_engine.generate(prompts, sp)
        except Exception as _vllm_err:
            print(f"WARNING: vLLM batch failed ({_vllm_err}), falling back to per-row inference")
            outputs = []
            for _i, _p in enumerate(prompts):
                try:
                    _out = _llm_engine.generate([_p], sp)
                    outputs.append(_out[0])
                except Exception as _row_err:
                    print(f"  Row {_i} failed: {_row_err}")
                    outputs.append(None)
        print(f"  {_batch_label} done in {time.perf_counter()-_vllm_t:.0f}s")

        records = []
        failed = 0
        failures_log = []
        pbar = tqdm(outputs, desc="Parsing outputs", ncols=90, unit="row")
        for i, out in enumerate(pbar):
            if out is None:
                failed += 1
                failures_log.append({"idx": i, "error": "per-row LLM inference failed", "raw": ""})
                records.append({"vibe_sentence": None})
                pbar.set_postfix(failed=failed)
                continue
            if not out.outputs:
                failed += 1
                failures_log.append({"idx": i, "error": "empty LLM output (no outputs[])", "raw": ""})
                records.append({"vibe_sentence": None})
                pbar.set_postfix(failed=failed)
                continue
            text = out.outputs[0].text.strip()
            try:
                obj = EnrichmentSchemaV2.model_validate_json(text)
                records.append(obj.model_dump())
            except Exception:
                try:
                    obj = EnrichmentSchemaV2.model_validate_json(repair_json(text))
                    records.append(obj.model_dump())
                except Exception as e2:
                    failed += 1
                    failures_log.append({"idx": i, "error": str(e2), "raw": text[:200]})
                    records.append({"vibe_sentence": None})
            pbar.set_postfix(failed=failed)
        pbar.close()

        if failures_log:
            fail_path = str(TIER_C_PATH) + ".failures.jsonl"
            with open(fail_path, "w") as ff:
                for entry in failures_log:
                    ff.write(_json.dumps(entry) + "\n")
            print(f"{failed} failures logged to {fail_path}")

        # Merge parsed fields back into df_500
        enriched_c = df_500.copy()
        for key in EnrichmentSchemaV2.model_fields.keys():
            enriched_c[key] = [r.get(key) for r in records]

    else:
        # -- Gemini / Outlines serial fallback --------------------------------
        from vibescents.enrich import enrich_dataframe, build_retrieval_text
        provider = "qwen" if USE_LOCAL_LLM else "gemini"
        enriched_c = enrich_dataframe(
            df_500,
            provider=provider,
            max_rows=500,
            checkpoint_path=TIER_C_CKPT,
            failures_path=str(TIER_C_PATH) + ".failures.jsonl",
            batch_size=32,
        )

    from vibescents.enrich import build_retrieval_text
    enriched_c = build_retrieval_text(enriched_c)

    # Background save so we can proceed
    def _save_c(df=enriched_c, path=TIER_C_PATH):
        df.to_csv(path, index=False)
        print(f"  [bg] Saved TIER C: {len(df)} rows -> {path}")

    _save_c_thread = threading.Thread(target=_save_c, daemon=False)
    _save_c_thread.start()

try:
    validate_enrichment(enriched_c)
except (AssertionError, ValueError) as e:
    print(f"WARNING: {e}")

print(f"TIER C enrichment | {len(enriched_c)} rows | {time.perf_counter()-_t0:.0f}s")




In [ ]:
# === Stage 9: Retrieval Comparison (TIER A raw) + Report ===
import json
import time
from tqdm.auto import tqdm
import numpy as np
from pathlib import Path
from vibescents.similarity import cosine_similarity_matrix, top_k_indices

_t0 = time.perf_counter()

# Load embeddings (wait for any bg saves if still running)
try:
    _bg_thread.join()
except NameError:
    pass
try:
    _save_c_thread.join()
except NameError:
    pass

occ_emb = np.load(Path(DRIVE_BASE) / "occasions" / "embeddings.npy")
raw_emb = np.load(Path(DRIVE_BASE) / "fragrance_raw_full" / "embeddings.npy")

print(f"Computing similarity matrix {occ_emb.shape} x {raw_emb.shape}...", end=" ", flush=True)
sim_matrix = cosine_similarity_matrix(occ_emb, raw_emb)   # (8, 35889)  --  vectorized matmul
print("done")

with open("examples/occasions.json") as f:
    occasions = json.load(f)

print("=== TIER A: Raw Corpus Top-5 per Occasion ===\n")
pbar = tqdm(enumerate(occasions), total=len(occasions), desc="Top-5 retrieval", ncols=90)
for i, occ in pbar:
    occ_text = occ["text"] if isinstance(occ, dict) else str(occ)
    top5_idx = top_k_indices(sim_matrix[i], k=5)
    pbar.set_postfix_str(occ_text[:40])
    print(f"\nOccasion: {occ_text}")
    for rank, idx in enumerate(top5_idx, 1):
        name  = df.iloc[idx].get("name", "?") if hasattr(df.iloc[idx], "get") else df.iloc[idx]["name"]
        brand = df.iloc[idx].get("brand", "?") if hasattr(df.iloc[idx], "get") else df.iloc[idx]["brand"]
        score = sim_matrix[i, idx]
        print(f"  {rank}. {brand}  --  {name}  (score: {score:.4f})")
pbar.close()

print(f"\nP0 STAGES COMPLETE | elapsed {time.perf_counter()-_t0:.0f}s")
print(f"Artifacts at: {DRIVE_BASE}")


---

## P1  --  Deep enrichment, RAW vs ENRICHED, Multimodal

In [ ]:
# === Stage 10: Enrich TIER B (top-2K rows)  --  vLLM batch inference ===
import time
import json
import threading
from tqdm.auto import tqdm
import pandas as pd
from pathlib import Path
from vibescents.week2_pipeline import select_tier_b, validate_enrichment

TIER_B_PATH = Path(DRIVE_BASE) / "vibescent_enriched_2k_v2.csv"
TIER_B_CKPT = str(TIER_B_PATH) + ".ckpt"
_t0 = time.perf_counter()

if TIER_B_PATH.exists():
    enriched_b = pd.read_csv(TIER_B_PATH)
    print(f"Loaded cached TIER B: {len(enriched_b)} rows")
else:
    tier_b_df = select_tier_b(df, target_size=2000, min_size=500)

    if (USE_LOCAL_LLM or USE_LIGHTWEIGHT_LLM) and globals().get("_llm_engine") and hasattr(globals()["_llm_engine"], "generate"):
        # -- vLLM batch (A100) or Gemma 4 E4B batch (T4) ----------------------
        from vibescents.enrich import _build_prompt
        from vibescents.schemas import EnrichmentSchemaV2
        from json_repair import repair_json
        if USE_LOCAL_LLM:
            from vllm import SamplingParams  # T4 uses shim defined in Cell 6
        import json as _json

        schema_dict = EnrichmentSchemaV2.model_json_schema()
        sp = SamplingParams(temperature=0.3, max_tokens=256, guided_json=schema_dict)

        print(f"Building {len(tier_b_df)} prompts...", end=" ", flush=True)
        prompts = [_build_prompt(row) for _, row in tier_b_df.iterrows()]
        print("done")

        _vllm_t = time.perf_counter()
        _batch_label = "vLLM" if USE_LOCAL_LLM else "Gemma"
        print(f"  {_batch_label} batch ({len(prompts)} rows) -- this may take several minutes...")
        try:
            outputs = _llm_engine.generate(prompts, sp)
        except Exception as _vllm_err:
            print(f"WARNING: vLLM batch failed ({_vllm_err}), falling back to per-row inference")
            outputs = []
            for _i, _p in enumerate(prompts):
                try:
                    _out = _llm_engine.generate([_p], sp)
                    outputs.append(_out[0])
                except Exception as _row_err:
                    print(f"  Row {_i} failed: {_row_err}")
                    outputs.append(None)
        print(f"  {_batch_label} done in {time.perf_counter()-_vllm_t:.0f}s")

        records = []
        failed = 0
        failures_log = []
        pbar = tqdm(outputs, desc="Parsing TIER B", ncols=90, unit="row")
        for i, out in enumerate(pbar):
            if out is None:
                failed += 1
                failures_log.append({"idx": i, "error": "per-row LLM inference failed", "raw": ""})
                records.append({"vibe_sentence": None})
                pbar.set_postfix(failed=failed)
                continue
            if not out.outputs:
                failed += 1
                failures_log.append({"idx": i, "error": "empty LLM output (no outputs[])", "raw": ""})
                records.append({"vibe_sentence": None})
                pbar.set_postfix(failed=failed)
                continue
            text = out.outputs[0].text.strip()
            try:
                obj = EnrichmentSchemaV2.model_validate_json(text)
                records.append(obj.model_dump())
            except Exception:
                try:
                    obj = EnrichmentSchemaV2.model_validate_json(repair_json(text))
                    records.append(obj.model_dump())
                except Exception as e2:
                    failed += 1
                    failures_log.append({"idx": i, "error": str(e2), "raw": text[:200]})
                    records.append({"vibe_sentence": None})
            pbar.set_postfix(failed=failed)
        pbar.close()

        if failures_log:
            fail_path = str(TIER_B_PATH) + ".failures.jsonl"
            with open(fail_path, "w") as ff:
                for entry in failures_log:
                    ff.write(_json.dumps(entry) + "\n")
            print(f"{failed} failures logged")

        enriched_b = tier_b_df.copy()
        for key in EnrichmentSchemaV2.model_fields.keys():
            enriched_b[key] = [r.get(key) for r in records]

    else:
        from vibescents.enrich import enrich_dataframe
        provider = "qwen" if USE_LOCAL_LLM else "gemini"
        enriched_b = enrich_dataframe(
            tier_b_df,
            provider=provider,
            max_rows=len(tier_b_df),
            checkpoint_path=TIER_B_CKPT,
            failures_path=str(TIER_B_PATH) + ".failures.jsonl",
            batch_size=32,
        )

    from vibescents.enrich import build_retrieval_text
    enriched_b = build_retrieval_text(enriched_b)

    def _save_b(df=enriched_b, path=TIER_B_PATH):
        df.to_csv(path, index=False)
        print(f"  [bg] Saved TIER B: {len(df)} rows -> {path}")

    _save_b_thread = threading.Thread(target=_save_b, daemon=False)
    _save_b_thread.start()

try:
    validate_enrichment(enriched_b)
except (AssertionError, ValueError) as e:
    print(f"WARNING: {e}")
print(f"TIER B: {len(enriched_b)} rows | {time.perf_counter()-_t0:.0f}s")





In [ ]:
# === Stage 11: Rebuild retrieval_text for enriched tiers ===
from tqdm.auto import tqdm
from vibescents.enrich import build_retrieval_text

pbar = tqdm(["TIER B", "TIER C"], desc="Stage 11  --  Build retrieval_text", ncols=90)
for label in pbar:
    pbar.set_postfix_str(label)
    if label == "TIER B":
        enriched_b = build_retrieval_text(enriched_b)
    else:
        enriched_c = build_retrieval_text(enriched_c)
pbar.close()

print(f"TIER B sample:\n{enriched_b['retrieval_text'].iloc[0][:200]}...")
print(f"\nTIER C sample:\n{enriched_c['retrieval_text'].iloc[0][:200]}...")


In [ ]:
# === Stage 12: Generate display_text for TIER B (optional stopgap) ===
print("Stage 12: display_text generation SKIPPED (optional  --  run separately if needed)")
print("Tip: use Qwen3.5-27B with a short 'tasting note' prompt schema when time permits")


In [ ]:
# === Stage 13: Unload LLM, Reload Qwen3-Embedding-8B ===
import os, time, gc, torch
from tqdm.auto import tqdm
from vibescents.week2_pipeline import ensure_model_loaded, mark_model_unloaded, check_disk_space

_t0 = time.perf_counter()
steps = ["Wait bg saves", "Unload LLM", "Clear VRAM", "Load embedder", "Verify"]
bar = tqdm(steps, desc="Stage 13  --  Reload embedder", ncols=90, unit="step")

# Wait for any background saves
for _thr_name in ("_save_b_thread", "_save_c_thread"):
    t = globals().get(_thr_name)
    if t is not None and hasattr(t, "is_alive") and t.is_alive():
        t.join()
bar.update(1)

_llm_engine = globals().get("_llm_engine")
if "llm_client" in dir():
    del llm_client
if _llm_engine is not None:
    del _llm_engine
gc.collect()
bar.update(1)

torch.cuda.empty_cache()
mark_model_unloaded()
bar.update(1)

bar.set_postfix_str("loading weights (sdpa)...")
from sentence_transformers import SentenceTransformer
model_kwargs = {"torch_dtype": "float16"}
if USE_INT8_EMBEDDER:
    model_kwargs = {"load_in_8bit": True}  # torch_dtype conflicts with bitsandbytes int8
else:
    model_kwargs["attn_implementation"] = "sdpa"

embedder = SentenceTransformer(
    "Qwen/Qwen3-Embedding-8B",
    trust_remote_code=True,
    model_kwargs=model_kwargs,
)
ensure_model_loaded("qwen3-embed-8b", lambda: None)
bar.update(1)

test_emb = embedder.encode(["preflight check"], normalize_embeddings=True, show_progress_bar=False)
assert test_emb.shape[1] == 4096
bar.update(1)
bar.close()

if os.path.exists("/content"):
    check_disk_space("/content")
print(f"Embedder reloaded | {time.perf_counter()-_t0:.0f}s")




In [ ]:
# === Stage 14: Embed Enriched TIER B + TIER C (batch=64, autocast) ===
PIPELINE_VERSION = globals().get("PIPELINE_VERSION", "v1")  # guard for re-run safety
# Parallel strategy: embed TIER B, start bg save, immediately embed TIER C
import time, threading, subprocess
import numpy as np
import torch
from tqdm.auto import tqdm
from pathlib import Path
from vibescents.week2_pipeline import (
    embedding_sanity_check, write_manifest, stage_complete,
)

ARTIFACTS_ENR_B = Path(DRIVE_BASE) / "fragrance_enriched_2k"
ARTIFACTS_ENR_C = Path(DRIVE_BASE) / "fragrance_enriched_500"
BATCH = 64
_t0 = time.perf_counter()
commit_sha = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()

def _encode_texts(texts, label):
    n_batches = (len(texts) + BATCH - 1) // BATCH
    parts = []
    pbar = tqdm(range(n_batches), desc=f"Embed {label}", ncols=100, unit="batch",
                postfix={"rows": 0})
    for bi in pbar:
        batch = texts[bi*BATCH : (bi+1)*BATCH]
        with torch.no_grad(), torch.amp.autocast('cuda'):
            raw = embedder.encode(batch, normalize_embeddings=False,
                                   show_progress_bar=False, convert_to_numpy=True)
        parts.append(raw[:, :1024].astype(np.float32))
        pbar.set_postfix(rows=f"{min((bi+1)*BATCH, len(texts)):,}")
    pbar.close()
    matrix = np.vstack(parts)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix / np.where(norms == 0, 1.0, norms)

# -- TIER B --------------------------------------------------------------------
if stage_complete("embed_enr_b", ARTIFACTS_ENR_B, PIPELINE_VERSION):
    enriched_b_emb = np.load(ARTIFACTS_ENR_B / "embeddings.npy")
    print(f"Cached TIER B: {enriched_b_emb.shape}")
else:
    texts_b = enriched_b["retrieval_text"].fillna("").tolist()
    enriched_b_emb = _encode_texts(texts_b, "TIER B enriched")
    ARTIFACTS_ENR_B.mkdir(parents=True, exist_ok=True)

    def _save_b_emb():
        np.save(ARTIFACTS_ENR_B / "embeddings.npy", enriched_b_emb)
        write_manifest(ARTIFACTS_ENR_B / "manifest.json", model="Qwen/Qwen3-Embedding-8B",
                       commit_sha=commit_sha, row_count=enriched_b_emb.shape[0],
                       dim=enriched_b_emb.shape[1], pipeline_version=PIPELINE_VERSION)
        print(f"  [bg] Saved TIER B enriched embeddings: {enriched_b_emb.shape}")

    _enr_b_thread = threading.Thread(target=_save_b_emb, daemon=False)
    _enr_b_thread.start()
    print(f"TIER B saved (bg) | {enriched_b_emb.shape}")

# -- TIER C (runs while TIER B saves in background) ----------------------------
if stage_complete("embed_enr_c", ARTIFACTS_ENR_C, PIPELINE_VERSION):
    enriched_c_emb = np.load(ARTIFACTS_ENR_C / "embeddings.npy")
    print(f"Cached TIER C: {enriched_c_emb.shape}")
else:
    texts_c = enriched_c["retrieval_text"].fillna("").tolist()
    enriched_c_emb = _encode_texts(texts_c, "TIER C enriched")
    ARTIFACTS_ENR_C.mkdir(parents=True, exist_ok=True)
    np.save(ARTIFACTS_ENR_C / "embeddings.npy", enriched_c_emb)
    write_manifest(ARTIFACTS_ENR_C / "manifest.json", model="Qwen/Qwen3-Embedding-8B",
                   commit_sha=commit_sha, row_count=enriched_c_emb.shape[0],
                   dim=enriched_c_emb.shape[1], pipeline_version=PIPELINE_VERSION)
    print(f"TIER C enriched embeddings: {enriched_c_emb.shape}")

# Wait for bg save before sanity check
try:
    _enr_b_thread.join()
except NameError:
    pass

embedding_sanity_check(enriched_b_emb)
print(f"Sanity check passed | elapsed {time.perf_counter()-_t0:.0f}s")


In [ ]:
# === Stage 15: RAW vs ENRICHED Retrieval Comparison ===
import time
import json
import numpy as np
from tqdm.auto import tqdm
from pathlib import Path
from vibescents.similarity import cosine_similarity_matrix, top_k_indices

_t0 = time.perf_counter()
occ_emb      = np.load(Path(DRIVE_BASE) / "occasions" / "embeddings.npy")
raw_emb      = np.load(Path(DRIVE_BASE) / "fragrance_raw_full" / "embeddings.npy")

# Align TIER C raw embeddings using fragrance_id (safe against non-prefix subsets)
enr_500_emb  = np.load(Path(DRIVE_BASE) / "fragrance_enriched_500" / "embeddings.npy")
enriched_c   = globals().get("enriched_c")
if (enriched_c is not None and "fragrance_id" in df.columns
        and "fragrance_id" in enriched_c.columns):
    _fid_to_raw = {str(fid): i for i, fid in enumerate(df["fragrance_id"])}
    _tier_c_raw_idx = [_fid_to_raw.get(str(fid)) for fid in enriched_c["fragrance_id"]]
    _valid = [(ci, ri) for ci, ri in enumerate(_tier_c_raw_idx) if ri is not None]
    if _valid:
        _enr_idx, _raw_idx = zip(*_valid)
        enr_500_emb = enr_500_emb[list(_enr_idx)]
        raw_500_emb = raw_emb[list(_raw_idx)]
    else:
        raw_500_emb = raw_emb[:len(enr_500_emb)]
else:
    raw_500_emb = raw_emb[:len(enr_500_emb)]

print("Computing similarity matrices (vectorized)...", end=" ", flush=True)
sim_raw = cosine_similarity_matrix(occ_emb, raw_500_emb)
sim_enr = cosine_similarity_matrix(occ_emb, enr_500_emb)
print("done")

with open("examples/occasions.json") as f:
    occasions = json.load(f)

comparison_lines = ["=== TIER C: RAW vs ENRICHED Top-5 per Occasion ===\n"]
different_count = 0

pbar = tqdm(enumerate(occasions), total=len(occasions), desc="RAW vs ENRICHED", ncols=90)
for i, occ in pbar:
    occ_text = occ["text"] if isinstance(occ, dict) else str(occ)
    top5_raw = set(top_k_indices(sim_raw[i], k=5).tolist())
    top5_enr = set(top_k_indices(sim_enr[i], k=5).tolist())
    overlap  = len(top5_raw & top5_enr)
    if overlap < 5:
        different_count += 1
    comparison_lines.append(
        f"Occasion: {occ_text} | Overlap: {overlap}/5 | Different: {5-overlap}"
    )
    pbar.set_postfix(diff=different_count)
pbar.close()

comparison_lines.append(
    f"\nOccasions with >=1 different result: {different_count}/{len(occasions)}"
)
comparison_text = "\n".join(comparison_lines)
print(comparison_text)

comp_path = Path(DRIVE_BASE) / "retrieval_comparison.txt"
comp_path.write_text(comparison_text)
print(f"\nSaved to {comp_path} | {time.perf_counter()-_t0:.1f}s")



In [ ]:
# === Stage 16: Load Qwen3-VL-Embedding-8B ===
import time, gc, torch
from tqdm.auto import tqdm
from vibescents.week2_pipeline import ensure_model_loaded, mark_model_unloaded, check_disk_space

_t0 = time.perf_counter()

if not USE_LOCAL_MM:
    print("Stage 16: SKIPPED (no local multimodal on this GPU tier)")
    vl_embedder = None
else:
    steps = ["Unload text embedder", "Clear VRAM", "Load VL model", "Verify"]
    bar = tqdm(steps, desc="Stage 16  --  Load VL embedder", ncols=90, unit="step")

    if globals().get("embedder") is not None:
        del embedder
        embedder = None
    gc.collect()
    bar.update(1)

    torch.cuda.empty_cache()
    mark_model_unloaded()
    bar.update(1)

    _vl_prec = "int8" if USE_INT8_VL else "fp16"
    bar.set_postfix_str(f"loading Qwen3-VL-Embedding-8B ({_vl_prec})...")
    from vibescents.embeddings import Qwen3VLMultimodalEmbedder
    vl_embedder = Qwen3VLMultimodalEmbedder(load_in_8bit=USE_INT8_VL)
    ensure_model_loaded("qwen3-vl-embed-8b", lambda: None)
    bar.update(1)

    # Quick pre-flight
    _test = vl_embedder.embed_multimodal_documents(["test fragrance text"])
    assert _test.shape[1] >= 1024, f"VL output dim too small: {_test.shape}"
    bar.update(1)
    bar.close()

    if os.path.exists("/content"):
        check_disk_space("/content")
    print(f"Qwen3-VL-Embedding-8B ready | {time.perf_counter()-_t0:.0f}s")





In [ ]:
# === Stage 17: Multimodal Embed TIER B Documents (batch=16, background save) ===
PIPELINE_VERSION = globals().get("PIPELINE_VERSION", "v1")  # guard for re-run safety
import time, threading, subprocess
import numpy as np
from tqdm.auto import tqdm
from pathlib import Path
from vibescents.week2_pipeline import write_manifest, stage_complete

ARTIFACTS_MM = Path(DRIVE_BASE) / "multimodal_2k"
BATCH_MM = 16   # 2x original; VL model heavier but A100 has headroom
_t0 = time.perf_counter()

if vl_embedder is None:
    print("Stage 17: SKIPPED (no multimodal embedder)")
    mm_doc_emb = None
elif stage_complete("mm_embed_b", ARTIFACTS_MM, PIPELINE_VERSION):
    mm_doc_emb = np.load(ARTIFACTS_MM / "doc_embeddings.npy")
    print(f"Loaded cached MM doc embeddings: {mm_doc_emb.shape}")
else:
    texts_b = enriched_b["retrieval_text"].fillna("").tolist()
    n_batches = (len(texts_b) + BATCH_MM - 1) // BATCH_MM
    parts = []

    pbar = tqdm(range(n_batches), desc="Stage 17  --  MM embed TIER B",
                ncols=100, unit="batch", total=n_batches)
    for bi in pbar:
        batch = texts_b[bi*BATCH_MM : (bi+1)*BATCH_MM]
        emb = vl_embedder.embed_multimodal_documents(batch)
        parts.append(emb[:, :1024].astype(np.float32))
        pbar.set_postfix(rows=f"{min((bi+1)*BATCH_MM, len(texts_b)):,}")

        # Checkpoint every 50 batches (~=800 rows)
        if (bi + 1) % 50 == 0:
            _partial = np.vstack(parts)
            _ckpt = ARTIFACTS_MM / f"doc_partial_{bi}.npy"
            ARTIFACTS_MM.mkdir(parents=True, exist_ok=True)
            np.save(_ckpt, _partial)
    pbar.close()

    mm_doc_emb = np.vstack(parts).astype(np.float32)
    norms = np.linalg.norm(mm_doc_emb, axis=1, keepdims=True)
    mm_doc_emb = mm_doc_emb / np.where(norms == 0, 1.0, norms)

    ARTIFACTS_MM.mkdir(parents=True, exist_ok=True)
    commit_sha = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()

    def _save_mm():
        np.save(ARTIFACTS_MM / "doc_embeddings.npy", mm_doc_emb)
        write_manifest(ARTIFACTS_MM / "manifest.json",
                       model="Qwen/Qwen3-VL-Embedding-8B",
                       commit_sha=commit_sha,
                       row_count=mm_doc_emb.shape[0],
                       dim=mm_doc_emb.shape[1],
                       pipeline_version=PIPELINE_VERSION)
        print(f"  [bg] Saved MM doc embeddings: {mm_doc_emb.shape}")

    _mm_save_thread = threading.Thread(target=_save_mm, daemon=False)
    _mm_save_thread.start()
    print(f"MM doc embeddings computed | {mm_doc_emb.shape} | {time.perf_counter()-_t0:.0f}s")



In [ ]:
# === Stage 18: Multimodal Query Probes ===
import time
import json
import numpy as np
from tqdm.auto import tqdm
from vibescents.similarity import cosine_similarity_matrix, top_k_indices

_t0 = time.perf_counter()

if vl_embedder is None or mm_doc_emb is None:
    print("Stage 18: SKIPPED (no multimodal embedder)")
else:
    # Wait for background MM save
    try:
        _mm_save_thread.join()
    except NameError:
        pass

    with open("examples/occasions.json") as f:
        occasions = json.load(f)
    query_texts = [occ["text"] if isinstance(occ, dict) else str(occ) for occ in occasions]

    print("Embedding query texts via VL model...", end=" ", flush=True)
    query_emb = vl_embedder.embed_multimodal_documents(query_texts)
    query_emb = query_emb[:, :1024].astype(np.float32)
    norms = np.linalg.norm(query_emb, axis=1, keepdims=True)
    query_emb = query_emb / np.where(norms == 0, 1.0, norms)
    print("done")

    mm_sim = cosine_similarity_matrix(query_emb, mm_doc_emb)

    print("=== Multimodal Text-Only Query Top-5 ===\n")
    pbar = tqdm(enumerate(occasions), total=len(occasions), desc="MM queries", ncols=90)
    for i, occ in pbar:
        occ_text = occ["text"] if isinstance(occ, dict) else str(occ)
        top5 = top_k_indices(mm_sim[i], k=5)
        pbar.set_postfix_str(occ_text[:40])
        print(f"\nQuery: {occ_text}")
        for rank, idx in enumerate(top5, 1):
            row = enriched_b.iloc[idx]
            name  = row.get("name", "?") if hasattr(row, "get") else row["name"]
            brand = row.get("brand", "?") if hasattr(row, "get") else row["brand"]
            print(f"  {rank}. {brand}  --  {name}  (score: {mm_sim[i, idx]:.4f})")
    pbar.close()
    print(f"\nStage 18 complete | {time.perf_counter()-_t0:.1f}s")



In [ ]:
# === Stage 19: Sanity Checks ===
import numpy as np
from tqdm.auto import tqdm
from pathlib import Path
from vibescents.week2_pipeline import read_manifest

checks = [
    ("Raw full",     Path(DRIVE_BASE) / "fragrance_raw_full",     "embeddings.npy"),
    ("Occasions",    Path(DRIVE_BASE) / "occasions",               "embeddings.npy"),
    ("Enriched 2K",  Path(DRIVE_BASE) / "fragrance_enriched_2k",  "embeddings.npy"),
    ("Enriched 500", Path(DRIVE_BASE) / "fragrance_enriched_500", "embeddings.npy"),
    ("MM 2K",        Path(DRIVE_BASE) / "multimodal_2k",          "doc_embeddings.npy"),
]

print("=== Artifact Sanity Checks ===\n")
pbar = tqdm(checks, desc="Checking artifacts", ncols=90, unit="artifact")
all_ok = True
for name, path, emb_file in pbar:
    pbar.set_postfix_str(name)
    manifest_path = path / "manifest.json"
    if not manifest_path.exists():
        print(f"  {name}: NO MANIFEST")
        continue
    m = read_manifest(manifest_path)
    emb_path = path / emb_file
    if emb_path.exists():
        shape = np.load(emb_path, mmap_mode="r").shape
        ok = shape[0] == m["row_count"] and shape[1] == m["dim"]
        status = "OK" if ok else f"MISMATCH (file={shape}, manifest=({m['row_count']},{m['dim']}))"
        if not ok:
            all_ok = False
    else:
        status = "NO EMBEDDING FILE"
        all_ok = False
    print(f"  {name}: {status} | v={m['pipeline_version']}")
pbar.close()

print(f"\n{'All checks PASSED' if all_ok else 'SOME CHECKS FAILED  --  see above'}")


In [ ]:
# === Stage 20: Report Writer ===
from pathlib import Path

report_path = Path("results/week2_report.md")
if report_path.exists():
    content = report_path.read_text()
    if content.rstrip().endswith("`") and not content.rstrip().endswith("```"):
        content = content.rstrip()[:-1] + "\n"
        report_path.write_text(content)
        print("Fixed trailing backtick in week2_report.md")
    else:
        print("No trailing backtick issue detected")
    print(f"Report at {report_path} ({len(content)} chars)")
else:
    print("results/week2_report.md not found  --  populate manually from artifacts above")

print("Stage 20: Review artifacts and fill report sections 2-7 based on cell outputs above.")


In [ ]:
# === Stage 21: Triple Artifact Sink (parallel copy) ===
import shutil
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from pathlib import Path

_t0 = time.perf_counter()

# 1. Drive  --  already saved throughout (DRIVE_BASE)
print(f"Drive artifacts at: {DRIVE_BASE}")

# 2. Repo artifacts  --  copy small files only (<20 MB)
repo_artifacts = Path("artifacts")
repo_artifacts.mkdir(exist_ok=True)

copy_tasks = []

# Occasions (tiny, ~100 KB)
occ_src = Path(DRIVE_BASE) / "occasions"
if occ_src.exists():
    occ_dst = repo_artifacts / "occasions"
    occ_dst.mkdir(parents=True, exist_ok=True)
    for f in occ_src.iterdir():
        if f.stat().st_size < 20_000_000:
            copy_tasks.append((f, occ_dst / f.name, f.name))

# Small JSON / text artifacts
for fname, dst_name in [
    ("benchmark_cases.json",     "benchmark_cases.json"),
    ("retrieval_comparison.txt", "retrieval_comparison.txt"),
]:
    src = Path(DRIVE_BASE) / fname
    if src.exists():
        copy_tasks.append((src, repo_artifacts / dst_name, dst_name))

def _copy(task):
    src, dst, label = task
    shutil.copy2(src, dst)
    return label

pbar = tqdm(total=len(copy_tasks), desc="Stage 21  --  Copy artifacts", ncols=90, unit="file")
with ThreadPoolExecutor(max_workers=4) as pool:
    futures = {pool.submit(_copy, t): t[2] for t in copy_tasks}
    for fut in as_completed(futures):
        label = futures[fut]
        try:
            fut.result()
            pbar.set_postfix_str(label)
        except Exception as e:
            print(f"  WARNING: failed to copy {label}: {e}")
        pbar.update(1)
pbar.close()

# 3. Git add snippet
print("\n=== Git Add Snippet (large .npy excluded) ===")
print("git add artifacts/occasions/ artifacts/benchmark_cases.json artifacts/retrieval_comparison.txt")
print("git add notebooks/harsh_week2_pipeline.ipynb notebooks/harsh_week3_reranker.ipynb notebooks/harsh_week4_demo.ipynb")
print("git add src/vibescents/ tests/")

# 4. Colab download
try:
    from google.colab import files
    _dl_targets = [
        Path(DRIVE_BASE) / "benchmark_cases.json",
        repo_artifacts / "retrieval_comparison.txt",
    ]
    _dl_bar = tqdm(_dl_targets, desc="Downloading key files", ncols=90)
    for p in _dl_bar:
        if p.exists():
            _dl_bar.set_postfix_str(p.name)
            files.download(str(p))
    _dl_bar.close()
except ImportError:
    pass

print(f"\n=== WEEK 2 PIPELINE COMPLETE === elapsed {time.perf_counter()-_t0:.1f}s")
print(f"All artifacts at: {DRIVE_BASE}")
